### World Heritage Parser

In [5]:
import xml.etree.ElementTree as ET
import csv
import re
import os

def clean_text(raw_html):
    if not raw_html:
        return "暂无描述"
    # 清洗掉 HTML 标签
    cleanr = re.compile('<.*?>')
    cleantext = re.sub(cleanr, '', str(raw_html))
    return cleantext.replace('\n', ' ').strip()

def main():
    if not os.path.exists('unesco.xml'):
        print("错误：找不到 unesco.xml 文件！")
        return

    print("正在解析本地的 unesco.xml 文件...")
    
    try:
        tree = ET.parse('unesco.xml')
        root = tree.getroot()
    except Exception as e:
        print(f"解析 XML 文件失败，请检查文件是否完整: {e}")
        return
    
    csv_data = []
    
    # 找到所有的 row 节点
    rows = root.findall('.//row')
    
    for row in rows:
        # 1. 获取名字和国家 (UNESCO 新标签名: site 和 states)
        site_node = row.find('site')
        site_name = site_node.text if site_node is not None else "Unknown Site"
        
        country_node = row.find('states')
        country = country_node.text if country_node is not None else "Unknown Country"
        
        full_name = f"{country} - {site_name}".replace(',', '，')
        
        # 2. 获取经纬度 (UNESCO 把它们藏深了，我们用 .// 深度查找)
        lat_node = row.find('.//latitude')
        lng_node = row.find('.//longitude')
        lat = lat_node.text if lat_node is not None else ""
        lng = lng_node.text if lng_node is not None else ""
        
        # 3. 获取描述 (UNESCO 新标签名: short_description)
        desc_node = row.find('short_description')
        desc_raw = desc_node.text if desc_node is not None else ""
        desc = clean_text(desc_raw).replace(',', '，')
        
        # 4. 获取链接
        id_node = row.find('id_number')
        site_id = id_node.text if id_node is not None else ""
        link = f"https://whc.unesco.org/en/list/{site_id}" if site_id else "https://whc.unesco.org/"
        
        # 5. 分类映射
        cat_node = row.find('category')
        category = cat_node.text if cat_node is not None else ""
        if category == 'Cultural':
            site_type = 'building'
        elif category == 'Natural':
            site_type = 'nature'
        else:
            site_type = 'mountain'
            
        # 只要经纬度不是空的，就抓取下来
        if lat and lng:
            csv_data.append([full_name, lat, lng, desc, link, site_type])

    output_filename = 'landmarks.csv'
    with open(output_filename, 'w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(['name', 'lat', 'lng', 'desc', 'link', 'type'])
        writer.writerows(csv_data)
        
    print(f"大功告成！已成功提取 {len(csv_data)} 个世界遗产，并保存至 {output_filename}。")

if __name__ == "__main__":
    main()

正在解析本地的 unesco.xml 文件...
大功告成！已成功提取 1245 个世界遗产，并保存至 landmarks.csv。


In [6]:
!pip3.10 install deep-translator

### Translator

In [7]:
import xml.etree.ElementTree as ET
import csv
import re
import os
from deep_translator import GoogleTranslator

def clean_text(raw_html):
    if not raw_html: return "暂无描述"
    cleanr = re.compile('<.*?>')
    return re.sub(cleanr, '', str(raw_html)).replace('\n', ' ').strip()

def main():
    if not os.path.exists('unesco.xml'):
        print("请确保 unesco.xml 文件在同目录下！")
        return

    print("正在解析并翻译数据（这需要几分钟时间，请耐心等待）...")
    tree = ET.parse('unesco.xml')
    root = tree.getroot()
    
    # 初始化翻译器
    translator = GoogleTranslator(source='en', target='zh-CN')
    csv_data = []
    
    rows = root.findall('.//row')
    total = len(rows)
    
    for idx, row in enumerate(rows):
        site_node = row.find('site')
        site_en = site_node.text if site_node is not None else ""
        
        country_node = row.find('states')
        country_en = country_node.text if country_node is not None else ""
        
        lat_node = row.find('.//latitude')
        lng_node = row.find('.//longitude')
        lat = lat_node.text if lat_node is not None else ""
        lng = lng_node.text if lng_node is not None else ""
        
        desc_node = row.find('short_description')
        desc_en = clean_text(desc_node.text if desc_node is not None else "")
        
        if not (lat and lng and site_en): continue

        # 核心：将名字和描述翻译成中文 (遇到报错跳过翻译)
        try:
            site_zh = translator.translate(site_en)
            desc_zh = translator.translate(desc_en[:500]) + "..." # 截取前500字符翻译提升速度
            country_zh = translator.translate(country_en.split(',')[0]) # 只取第一个国家名
        except:
            site_zh, desc_zh, country_zh = site_en, desc_en, country_en
            
        full_name_zh = f"{country_zh} - {site_zh}".replace(',', '，')
        
        id_node = row.find('id_number')
        site_id = id_node.text if id_node is not None else ""
        link = f"https://whc.unesco.org/en/list/{site_id}"
        
        cat_node = row.find('category')
        category = cat_node.text if cat_node is not None else ""
        site_type = 'building' if category == 'Cultural' else ('nature' if category == 'Natural' else 'mountain')
        
        # 注意：这里我们多加了一列 country_en，这是为了前端地图匹配用的！
        csv_data.append([full_name_zh, lat, lng, desc_zh, link, site_type, country_en])
        
        if (idx + 1) % 100 == 0:
            print(f"已处理 {idx + 1} / {total} 条...")

    with open('landmarks.csv', 'w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(['name', 'lat', 'lng', 'desc', 'link', 'type', 'country_en'])
        writer.writerows(csv_data)
        
    print(f"翻译完成！成功保存 {len(csv_data)} 条中文数据到 landmarks.csv。")

if __name__ == "__main__":
    main()

正在解析并翻译数据（这需要几分钟时间，请耐心等待）...
已处理 100 / 1248 条...
已处理 200 / 1248 条...
已处理 300 / 1248 条...
已处理 400 / 1248 条...
已处理 500 / 1248 条...
已处理 600 / 1248 条...
已处理 700 / 1248 条...
已处理 800 / 1248 条...
已处理 900 / 1248 条...
已处理 1000 / 1248 条...
已处理 1100 / 1248 条...
已处理 1200 / 1248 条...
翻译完成！成功保存 1245 条中文数据到 landmarks.csv。


### Macro_data Crawler

In [1]:
!pip3.10 install wbgapi

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [wbgapi]


In [ ]:
import pandas as pd
import requests
import os
import time

# 1. 基础配置
os.makedirs('data', exist_ok=True)
OUTPUT_PATH = 'data/macro_data.csv'

# 国家列表映射 (按你提供的列表)
COUNTRY_MAP = {
    "China": "中国", "Japan": "日本", "South Korea": "韩国", "North Korea": "朝鲜", "India": "印度", 
    "Vietnam": "越南", "Thailand": "泰国", "Malaysia": "马来西亚", "Singapore": "新加坡", 
    "Indonesia": "印度尼西亚", "Philippines": "菲律宾", "Pakistan": "巴基斯坦", "Bangladesh": "孟加拉国", 
    "Sri Lanka": "斯里兰卡", "Myanmar": "缅甸", "Cambodia": "柬埔寨", "Laos": "老挝", "Nepal": "尼泊尔", 
    "Mongolia": "蒙古", "Kazakhstan": "哈萨克斯坦", "Uzbekistan": "乌兹别克斯坦", "Turkmenistan": "土库曼斯坦", 
    "Kyrgyzstan": "吉尔吉斯斯坦", "Tajikistan": "塔吉克斯坦", "Iran": "伊朗", "Iraq": "伊拉克", 
    "Saudi Arabia": "沙特阿拉伯", "United Arab Emirates": "阿联酋", "Turkey": "土耳其", "Israel": "以色列", 
    "Palestine": "巴勒斯坦", "Jordan": "约旦", "Syria": "叙利亚", "Lebanon": "黎巴嫩", "Kuwait": "科威特", 
    "Oman": "阿曼", "Qatar": "卡塔尔", "Yemen": "也门", "Afghanistan": "阿富汗", "Georgia": "格鲁吉亚", 
    "Armenia": "亚美尼亚", "Azerbaijan": "阿塞拜疆", "Greenland": "格陵兰岛", "Russia": "俄罗斯", 
    "Germany": "德国", "France": "法国", "United Kingdom": "英国", "Italy": "意大利", "Spain": "西班牙", 
    "Portugal": "葡萄牙", "Netherlands": "荷兰", "Belgium": "比利时", "Switzerland": "瑞士", 
    "Austria": "奥地利", "Greece": "希腊", "Sweden": "瑞典", "Norway": "挪威", "Denmark": "丹麦", 
    "Finland": "芬兰", "Ireland": "爱尔兰", "Poland": "波兰", "Czech Republic": "捷克", 
    "Slovakia": "斯洛伐克", "Hungary": "匈牙利", "Romania": "罗马尼亚", "Bulgaria": "保加利亚", 
    "Serbia": "塞尔维亚", "Croatia": "克罗地亚", "Slovenia": "斯洛文尼亚", "Bosnia and Herzegovina": "波黑", 
    "Montenegro": "黑山", "North Macedonia": "北马其顿", "Albania": "阿尔巴尼亚", "Ukraine": "乌克兰", 
    "Belarus": "白俄罗斯", "Estonia": "爱沙尼亚", "Latvia": "拉脱维亚", "Lithuania": "立陶宛", 
    "Moldova": "摩尔多瓦", "Iceland": "冰岛", "Luxembourg": "卢森堡", "Monaco": "摩纳哥", "Malta": "马耳他", 
    "Belize": "伯利兹", "United States of America": "美国", "Canada": "加拿大", "Mexico": "墨西哥", 
    "Brazil": "巴西", "Argentina": "阿根廷", "Chile": "智利", "Colombia": "哥伦比亚", "Peru": "秘鲁", 
    "Venezuela": "委内瑞拉", "Ecuador": "厄瓜多尔", "Bolivia": "玻利维亚", "Paraguay": "巴拉圭", 
    "Urguay": "乌拉圭", "Cuba": "古巴", "Jamaica": "牙买加", "Panama": "巴拿马", "Costa Rica": "哥斯达黎加", 
    "Guatemala": "危地马拉", "Honduras": "洪都拉斯", "Dominican Republic": "多米尼加", "Australia": "澳大利亚", 
    "New Zealand": "新西兰", "Papua New Guinea": "巴布亚新几内亚", "Fiji": "斐济", "Central African Republic": "中非共和国", 
    "Ivory Coast": "科特迪瓦", "Benin": "贝宁", "Burundi": "布隆迪", "Egypt": "埃及", "South Africa": "南非", 
    "Nigeria": "尼日利亚", "Ethiopia": "埃塞俄比亚", "Kenya": "肯尼亚", "Algeria": "阿尔及利亚", 
    "Morocco": "摩洛哥", "Libya": "利比亚", "Sudan": "苏丹", "Tunisia": "突尼斯", "Ghana": "加纳", 
    "Cote d'Ivoire": "科特迪瓦", "Senegal": "塞内加尔", "Democratic Republic of the Congo": "刚果金", 
    "Republic of the Congo": "刚果布", "Angola": "安哥拉", "Tanzania": "坦桑尼亚", "Uganda": "乌干达", 
    "Zambia": "赞比亚", "Zimbabwe": "津巴布韦", "Madagascar": "马达加斯加", "Mauritius": "毛里求斯", 
    "Antarctica": "南极洲", "French Southern and Antarctic Lands": "法属南半球和南极领地", "Bermuda": "百慕大", 
    "Burkina Faso": "布基纳法索", "The Bahamas": "巴哈马", "Bhutan": "不丹", "Djibouti": "吉布提", 
    "Falkland Islands": "福克兰群岛", "Cameroon": "喀麦隆", "Botswana": "博茨瓦纳", "Northern Cyprus": "北塞浦路斯", 
    "Cyprus": "塞浦路斯", "Equatorial Guinea": "赤道几内亚", "Mozambique": "莫桑比克", "Kosovo": "科索沃", 
    "Malawi": "马拉维", "New Caledonia": "新喀里多尼亚"
}

# 针对不同数据源的名称转换
NAME_FIX = {
    "United States of America": "United States",
    "South Korea": "Korea, Rep.",
    "Russia": "Russian Federation",
    "Turkey": "Turkiye",
    "Egypt": "Egypt, Arab Rep.",
    "Iran": "Iran, Islamic Rep.",
    "Venezuela": "Venezuela, RB",
    "Slovakia": "Slovak Republic"
}

INDICATORS = {
    'NY.GDP.MKTP.CD': 'gdp_total',         # 原始单位：USD
    'NY.GDP.PCAP.CD': 'gdp_per_capita',    # 原始单位：USD
    'NY.GDP.MKTP.KD.ZG': 'gdp_growth',      # 单位：%
    'EN.POP.DNST': 'pop_density',          # 单位：人/sq km
    'SI.POV.GINI': 'gini',
    'AG.LND.FRST.ZS': 'forest_area',       # 单位：%
    'EN.ATM.CO2E.PC': 'co2_per_capita',    # 单位：metric tons
    'IT.NET.USER.ZS': 'internet_user'      # 单位：%
}

def fetch_data(code):
    print(f"-> 正在获取 {INDICATORS[code]}...")
    url = f"https://api.worldbank.org/v2/country/all/indicator/{code}?format=json&per_page=400&mrv=1"
    try:
        r = requests.get(url, timeout=15)
        data = r.json()
        if len(data) > 1:
            return {item['country']['value']: item['value'] for item in data[1] if item['value'] is not None}
    except:
        return {}
    return {}

def main():
    # 1. 抓取所有原始数据
    raw_results = {}
    for code, field in INDICATORS.items():
        raw_results[field] = fetch_data(code)
        time.sleep(0.3)

    # 2. 处理与换算
    final_rows = []
    for eng_name in COUNTRY_MAP.keys():
        wb_name = NAME_FIX.get(eng_name, eng_name)
        
        # 处理 GDP 单位 (转为 10亿美元) 并保留两位小数
        gdp_val = raw_results['gdp_total'].get(wb_name)
        if gdp_val:
            gdp_val = round(gdp_val / 1_000_000_000, 2)

        row = {
            'place_id': eng_name,
            'gdp_total': gdp_val,
            'gdp_per_capita': round(raw_results['gdp_per_capita'].get(wb_name, 0), 2) if raw_results['gdp_per_capita'].get(wb_name) else None,
            'gdp_growth': round(raw_results['gdp_growth'].get(wb_name, 0), 2) if raw_results['gdp_growth'].get(wb_name) else None,
            'pop_density': round(raw_results['pop_density'].get(wb_name, 0), 2) if raw_results['pop_density'].get(wb_name) else None,
            'hdi': None,          # 静态预留
            'median_age': None,   # 静态预留
            'internet_user': round(raw_results['internet_user'].get(wb_name, 0), 2) if raw_results['internet_user'].get(wb_name) else None,
            'gini': round(raw_results['gini'].get(wb_name, 0), 2) if raw_results['gini'].get(wb_name) else None,
            'forest_area': round(raw_results['forest_area'].get(wb_name, 0), 2) if raw_results['forest_area'].get(wb_name) else None,
            'co2_per_capita': round(raw_results['co2_per_capita'].get(wb_name, 0), 2) if raw_results['co2_per_capita'].get(wb_name) else None,
            'avg_temp': None      # 静态预留
        }
        final_rows.append(row)

    # 3. 排序并保存
    df = pd.DataFrame(final_rows)
    df = df.sort_values('gdp_total', ascending=False, na_position='last')
    
    df.to_csv(OUTPUT_PATH, index=False, encoding='utf-8-sig')
    
    print("-" * 30)
    print(f"✅ 成功！数据已清洗完成。")
    print(f"📍 总GDP单位：10亿美元 (Billion USD)")
    print(f"📍 数值精度：保留2位小数")
    print(f"📍 文件位置：{OUTPUT_PATH}")

if __name__ == "__main__":
    main()

-> 正在获取 gdp_total...
-> 正在获取 gdp_per_capita...
-> 正在获取 gdp_growth...
-> 正在获取 pop_density...
-> 正在获取 gini...
-> 正在获取 forest_area...
-> 正在获取 co2_per_capita...
-> 正在获取 internet_user...
------------------------------
✅ 成功！数据已清洗完成。
📍 总GDP单位：10亿美元 (Billion USD)
📍 数值精度：保留2位小数
📍 文件位置：data/macro_data.csv


### Geographic place_id extraction

In [8]:
import json
import csv
import os

def extract_place_ids():
    # 定义文件路径
    world_json_path = 'data/world.json'
    china_json_path = 'data/china.json'
    output_csv_path = 'data/macro_data_new.csv'

    # 确保输出目录存在
    os.makedirs('data', exist_ok=True)

    place_ids = []

    # 1. 处理世界地图数据
    if os.path.exists(world_json_path):
        with open(world_json_path, 'r', encoding='utf-8') as f:
            world_data = json.load(f)
            for feature in world_data.get('features', []):
                name = feature.get('properties', {}).get('name')
                # 过滤掉世界地图中的中国，因为我们要用 china.json 的省级数据替代
                # 同时也过滤掉可能存在的 CHN 缩写
                if name and name not in ['China', 'CHN']:
                    place_ids.append(name)
        print(f"✅ 从 world.json 提取了 {len(place_ids)} 个国家/地区 ID")
    else:
        print(f"❌ 未找到 {world_json_path}")

    # 2. 处理中国省级数据
    china_count = 0
    if os.path.exists(china_json_path):
        # 强制添加 "China" 作为全国总体数据的 ID
        place_ids.append("China")
        
        with open(china_json_path, 'r', encoding='utf-8') as f:
            china_data = json.load(f)
            for feature in china_data.get('features', []):
                name = feature.get('properties', {}).get('name')
                if name:
                    place_ids.append(name)
                    china_count += 1
        print(f"✅ 从 china.json 提取了 {china_count} 个中国省份 ID（含 'China' 总项）")
    else:
        print(f"❌ 未找到 {china_json_path}")

    # 3. 写入新的 CSV 文件
    # 定义你想要的所有列名（根据你的 overview.js 配置）
    headers = [
        "place_id", "gdp_total", "gdp_per_capita", "gdp_growth", 
        "hdi", "pop_density", "median_age", "internet_user", 
        "gini", "forest_area", "co2_per_capita", "avg_temp"
    ]

    try:
        with open(output_csv_path, 'w', encoding='utf-8-sig', newline='') as f:
            writer = csv.DictWriter(f, fieldnames=headers)
            writer.writeheader()
            for pid in sorted(set(place_ids)): # 去重并排序
                # 初始化行，除 place_id 外其余为空
                row = {h: "" for h in headers}
                row["place_id"] = pid
                writer.writerow(row)
        print(f"\n🚀 成功！所有 ID 已填充至: {output_csv_path}")
        print(f"📊 总计地区数量: {len(set(place_ids))}")
    except Exception as e:
        print(f"❌ 写入 CSV 失败: {e}")

if __name__ == "__main__":
    extract_place_ids()

✅ 从 world.json 提取了 179 个国家/地区 ID
✅ 从 china.json 提取了 34 个中国省份 ID（含 'China' 总项）

🚀 成功！所有 ID 已填充至: data/macro_data_new.csv
📊 总计地区数量: 214


In [9]:
import pandas as pd
import requests
import time
from bs4 import BeautifulSoup

# --- 配置区 ---
OUTPUT_FILE = 'data/macro_data_new.csv'
# 世界银行指标映射 (Indicator Code)
WB_INDICATORS = {
    'gdp_total': 'NY.GDP.MKTP.CD',      # GDP (现价美元)
    'gdp_per_capita': 'NY.GDP.PCAP.CD', # 人均 GDP
    'gdp_growth': 'NY.GDP.MKTP.KD.ZG',  # GDP 增长率 (%)
    'pop_density': 'EN.POP.DNST',       # 人口密度
    'forest_area': 'AG.LND.FRST.ZS',    # 森林覆盖率 (%)
    'co2_per_capita': 'EN.ATM.CO2E.PC', # 人均碳排放
}

def fetch_world_bank_data():
    """获取全球国家数据"""
    print("正在从世界银行获取全球数据 (约需 30 秒)...")
    all_data = []
    
    for label, code in WB_INDICATORS.items():
        # 获取 2023 年数据（最新完整年份）
        url = f"https://api.worldbank.org/v2/country/all/indicator/{code}?format=json&per_page=300&date=2023"
        try:
            res = requests.get(url, timeout=15).json()
            if len(res) > 1:
                for item in res[1]:
                    country_name = item['country']['value']
                    val = item['value']
                    all_data.append({'place_id': country_name, 'indicator': label, 'value': val})
        except Exception as e:
            print(f"获取 {label} 失败: {e}")

    df = pd.DataFrame(all_data)
    # 透视表：将指标转为列
    df = df.pivot(index='place_id', columns='indicator', values='value').reset_index()
    
    # 单位换算：GDP 总额从“美元”换算为“十亿美元”
    if 'gdp_total' in df.columns:
        df['gdp_total'] = df['gdp_total'] / 1e9
        
    return df

def fetch_china_province_gdp():
    """爬取中国省级 GDP 数据 (示例来源: 维基百科/快易数据镜像)"""
    print("正在获取中国省级基础数据...")
    # 注意：实际生产环境建议下载统计局 Excel。这里演示爬取逻辑。
    # 假设我们从一个稳定的镜像源或硬编码一部分最新统计局数据
    provinces = [
        "广东", "江苏", "山东", "浙江", "河南", "四川", "湖北", "福建", "湖南", "安徽",
        "上海", "河北", "北京", "陕西", "江西", "辽宁", "重庆", "云南", "广西", "山西",
        "内蒙古", "贵州", "新疆", "天津", "黑龙江", "海南", "吉林", "甘肃", "青海", "宁夏", "西藏"
    ]
    
    china_data = []
    # 这里模拟填充（实际可用 requests 爬取具体的统计年鉴网页）
    for p in provinces:
        china_data.append({
            'place_id': f"{p}省" if p not in ["北京", "上海", "天津", "重庆", "内蒙古", "广西", "西藏", "新疆", "宁夏"] else p,
            'gdp_total': 0.0, # 需填入具体数值
            'gdp_per_capita': 0.0
        })
    return pd.DataFrame(china_data)

def main():
    # 1. 获取世界数据
    world_df = fetch_world_bank_data()
    
    # 2. 读取你之前生成的 place_id 列表
    try:
        base_df = pd.read_csv('data/macro_data_new.csv')
        # 仅保留 place_id 列，准备填充新数据
        base_df = base_df[['place_id']]
    except FileNotFoundError:
        print("未找到基础 CSV，将直接创建新表。")
        base_df = pd.DataFrame(columns=['place_id'])

    # 3. 合并数据
    # 以 base_df 为准，合并世界银行数据
    final_df = pd.merge(base_df, world_df, on='place_id', how='left')
    
    # 4. 格式化处理
    # 保留两位小数
    numeric_cols = final_df.select_dtypes(include=['number']).columns
    final_df[numeric_cols] = final_df[numeric_cols].round(2)
    
    # 5. 补充缺失列（确保 CSV 结构完整）
    required_columns = [
        "place_id", "gdp_total", "gdp_per_capita", "gdp_growth", 
        "hdi", "pop_density", "median_age", "internet_user", 
        "gini", "forest_area", "co2_per_capita", "avg_temp"
    ]
    for col in required_columns:
        if col not in final_df.columns:
            final_df[col] = ""
            
    # 重新排序列顺序
    final_df = final_df[required_columns]

    # 6. 保存
    final_df.to_csv(OUTPUT_FILE, index=False, encoding='utf-8-sig')
    print(f"\n✅ 数据处理完成！")
    print(f"📂 已保存至: {OUTPUT_FILE}")
    print(f"💡 提示：世界银行不提供中国各省数据，中国省级数据建议手动从《中国统计年鉴》填入。")

if __name__ == "__main__":
    main()

正在从世界银行获取全球数据 (约需 30 秒)...

✅ 数据处理完成！
📂 已保存至: data/macro_data_new.csv
💡 提示：世界银行不提供中国各省数据，中国省级数据建议手动从《中国统计年鉴》填入。


### 数据校验

In [ ]:
import pandas as pd
import numpy as np

def verify_macro_data(file_new, file_old):
    """
    比较两个 CSV 文件中 place_id 相同的非空数据是否一致
    """
    print(f"🔍 正在对比: {file_new} VS {file_old}\n")
    
    # 读取数据
    df_new = pd.read_csv(file_new).set_index('place_id')
    df_old = pd.read_csv(file_old).set_index('place_id')

    # 找出两个文件共同拥有的 place_id 和 列名
    common_ids = df_new.index.intersection(df_old.index)
    common_cols = df_new.columns.intersection(df_old.columns)

    mismatch_count = 0
    total_checks = 0

    for pid in common_ids:
        for col in common_cols:
            val_new = df_new.loc[pid, col]
            val_old = df_old.loc[pid, col]

            # 排除掉任何一方为 NaN (空值) 的情况，只检查“非空数据”
            if pd.notna(val_new) and pd.notna(val_old):
                total_checks += 1
                
                # 尝试转为浮点数比较，如果非数值则直接比较字符串
                try:
                    f_new, f_old = float(val_new), float(val_old)
                    # 使用 isclose 允许极小的浮点误差 (0.01 范围内)
                    is_same = np.isclose(f_new, f_old, atol=0.01)
                except ValueError:
                    is_same = str(val_new).strip() == str(val_old).strip()

                if not is_same:
                    mismatch_count += 1
                    print(f"⚠️ [不一致] 地区: {pid} | 指标: {col}")
                    print(f"   - 文件new: {val_new}")
                    print(f"   - 文件old: {val_old}")

    print("\n" + "="*30)
    if mismatch_count == 0:
        print(f"✅ 校验通过！共检查了 {total_checks} 个数据点，完全一致。")
    else:
        print(f"❌ 校验发现 {mismatch_count} 处差异（在总计 {total_checks} 个对比点中）。")

# 执行校验
if __name__ == "__main__":
    # 请确保文件路径正确
    try:
        verify_macro_data('data/macro_data_new.csv', 'data/macro_data.csv')
    except Exception as e:
        print(f"运行出错: {e}")

🔍 正在对比: data/macro_data_new.csv VS data/macro_data.csv

⚠️ [不一致] 地区: Canada | 指标: pop_density
   - 文件A: 39.46
   - 文件B: 4.56
⚠️ [不一致] 地区: Malaysia | 指标: pop_density
   - 文件A: 30.3
   - 文件B: 106.91

❌ 校验发现 2 处差异（在总计 609 个对比点中）。
